In [3]:
!pip install --upgrade onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 2.0 MB/s  0:00:08 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 1.9 MB/s  0:00:00? eta -:--:--
  Attempting uninstall: onnx
    Found existing installation: onnx 1.20.0
    Uninstalling onnx-1.20.0:━━━━━━━━━━━━━━━━━━━━━━━━━ 0/3 [onnx]
      Successfully uninstalled onnx-1.20.0━━━━━━━━ 0/3 [onnx]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [onnxscript]━━━━━━━ 2/3 [onnxscript]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [6]:
import torch
import torch.nn as nn
import onnxruntime as ort

# 1. Define a simple "Hello World" Model
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc = nn.Linear(10, 1) # Takes 10 inputs, gives 1 output

    def forward(self, x):
        return self.fc(x)

torch_model = SimpleModel()

# 2. Create "dummy" input (A 1x10 tensor of random numbers)
dummy_input = torch.randn(1, 10)

# 3. Export to ONNX format
torch.onnx.export(
    torch_model, 
    dummy_input, 
    "onnx_test.onnx", 
    dynamo=True  # <--- This is the key fix
)

[torch.onnx] Obtain model graph for `SimpleModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 20},
            producer_name='pytorch',
            producer_version='2.9.0.dev20250827',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"x"<FLOAT,[1,10]>
            ),
            outputs=(
                %"linear"<FLOAT,[1,1]>
            ),
            initializers=(
                %"fc.weight"<FLOAT,[1,10]>{TorchTensor<FLOAT,[1,10]>(Parameter containing: tensor([[-0.0613, -0.1989,  0.2578,  0.0842,  0.3131, -0.1183, -0.2994, -0.2848, 0.2464, -0.1124]], requires_grad=True), name='fc.weight')},
                %"fc.bias"<FLOAT,[1]>{TorchTensor<FLOAT,[1]>(Parameter containing: tensor([0.0077], requires_grad=True), name='fc.bias')}
            ),
        ) {
            0 |  # node_linear
                 %"linear"<FLOAT,[1,1]> ⬅️ ::Gemm(%"x", %"fc.weight"{[[-0.06131327897310257, -0.1989155

In [9]:
import onnxruntime as ort
import numpy as np

# 1. Load the ONNX model
session = ort.InferenceSession("onnx_test.onnx")

# 2. Prepare new data (using NumPy, not PyTorch!)
new_data = np.random.randn(1, 10).astype(np.float32)

# 3. Run the model
outputs = session.run(None, {'x': new_data}) 

print("ONNX Prediction:", outputs)

ONNX Prediction: [array([[-1.1291835]], dtype=float32)]
